# การตรวจจับไฟป่าจากภาพถ่ายทางอากาศด้วยคอมพิวเตอร์วิทัศน์ (computer vision) และอากาศยานไร้คนขับ (drone)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/main.ipynb)

**เอกสารเริ่มต้น** โน้ตบุ๊กฉบับนี้เป็นเอกสารนำทางของโครงการโดยรวม ครอบคลุมตั้งแต่วัตถุประสงค์ของระบบ
โครงสร้างและองค์ประกอบของระบบ ที่มาของชุดข้อมูล ไปจนถึงตำแหน่งของโค้ดที่ใช้งานได้จริงในแต่ละขั้นตอน

เอกสารฉบับนี้ทำหน้าที่เป็น *แผนที่* ส่วน [`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb) คือ *พื้นที่จริง*
ซึ่งรวมทั้งสามขั้นตอนไว้ในไฟล์เดียว และดำเนินการตามลำดับจากบนลงล่างได้ตลอดทั้งไฟล์

| ส่วน | เนื้อหา |
|---|---|
| [ส่วนที่ 1](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-1) | ฝึกโมเดลตรวจจับไฟด้วยชุดข้อมูลจาก Roboflow |
| [ส่วนที่ 2](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2) | ประมวลผลภาพนิ่งหนึ่งภาพด้วยโมเดลที่ฝึกแล้ว |
| [ส่วนที่ 3](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3) | ตรวจจับและติดตามวัตถุด้วย ByteTrack ตลอดทั้งวิดีโอ |

> **หมายเหตุ** ฉบับภาษาไทยในโฟลเดอร์ `th/` รวมทั้งสามส่วนไว้ในโน้ตบุ๊กไฟล์เดียว
> ขณะที่ฉบับภาษาอังกฤษในโฟลเดอร์หลักของโปรเจ็คนี้ (repo's root folder) ยังคงแยกเป็นสามไฟล์ ทั้งสองฉบับให้ผลการทำงานเหมือนกันทุกประการ
> ต่างกันเพียงการจัดวางไฟล์ คำอธิบาย และคอมเมนต์

ผู้อ่านควรศึกษาเอกสารฉบับนี้ตามลำดับจากต้นจนจบ โดยแต่ละหัวข้อจะระบุไว้ชัดเจนว่าควรเปิดโน้ตบุ๊กส่วนใดขึ้นมาดำเนินการประกอบ

*เอกสารฉบับนี้เรียบเรียงตามโครงการในบทความของ Roboflow เรื่อง
[Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/)
โดย Timothy M. (รายการอ้างอิงฉบับเต็มอยู่ท้ายเอกสาร) ทั้งนี้ ผู้เขียนปรับโค้ดทั้งหมดให้ทันสมัยแล้ว
เพราะบทความต้นฉบับเขียนขึ้นบนไลบรารีรุ่นปี 2023 ซึ่งใช้งานไม่ได้อีกต่อไป*

## เหตุผลของการตรวจจับไฟป่าจากทางอากาศ

การตรวจจับไฟป่าแต่เดิมอาศัยแนวทางหลักสองประการ ได้แก่ เซนเซอร์ภาคพื้นดินและภาพถ่ายดาวเทียม
ซึ่งต่างก็มีข้อจำกัดในตัวเอง เซนเซอร์ภาคพื้นดินตรวจวัดได้เฉพาะบริเวณที่ติดตั้งไว้เท่านั้น
และการติดตั้งให้ครอบคลุมทั้งผืนป่ามีต้นทุนสูงมาก ส่วนดาวเทียมครอบคลุมพื้นที่ได้กว้างกว่า
แต่ด้วยข้อจำกัดด้านความละเอียดเชิงพื้นที่ (spatial resolution) และรอบเวลาการถ่ายซ้ำ (revisit time)
จึงมักตรวจไม่พบไฟในช่วงที่ยังมีขนาดเล็กพอจะควบคุมได้ด้วยต้นทุนต่ำ

ช่วงเวลาดังกล่าวคือปัจจัยชี้ขาดของปัญหานี้ ไฟที่ตรวจพบภายในไม่กี่นาทีแรก
กับไฟกองเดียวกันที่ตรวจพบเมื่อเวลาผ่านไปหนึ่งชั่วโมง จัดเป็นคนละปัญหาโดยสิ้นเชิง

อากาศยานไร้คนขับที่ติดตั้งกล้องร่วมกับโมเดลคอมพิวเตอร์วิทัศน์จึงเข้ามาเติมเต็มช่องว่างระหว่างสองแนวทางข้างต้น
เนื่องจากสำรวจพื้นที่ได้รวดเร็วกว่าการเดินเท้าอย่างมาก บินในระดับต่ำพอที่จะเห็นรายละเอียดซึ่งดาวเทียมตรวจไม่พบ
และสามารถกำหนดตารางบินซ้ำเหนือภูมิประเทศที่ไม่สามารถจัดกำลังลาดตระเวนได้ทุกวัน

## องค์ประกอบของระบบ

ระบบนี้ประกอบด้วยองค์ประกอบสี่ส่วน แบ่งเป็นฮาร์ดแวร์สองส่วนและซอฟต์แวร์สองส่วน

| | องค์ประกอบ | หน้าที่ |
|---|---|---|
| 🛩️ | **อากาศยานไร้คนขับ** | นำเซนเซอร์ขึ้นบินเหนือภูมิประเทศที่การลาดตระเวนด้วยเท้าใช้เวลานานหรือมีความเสี่ยงสูง |
| 📷 | **โมดูลกล้อง WiFi** | บันทึกภาพและส่งสัญญาณกลับมายังสถานีภาคพื้นดิน |
| 🏷️ | **บัญชี Roboflow** | จัดเก็บชุดข้อมูล ตัดเฟรมภาพจากวิดีโอ และให้บริการเครื่องมือติดป้ายกำกับ |
| 📓 | **Google Colab** | ทรัพยากร GPU โดยไม่มีค่าใช้จ่ายสำหรับการฝึกโมเดล โน้ตบุ๊กทุกไฟล์ในคลังโค้ดนี้ทำงานบนแพลตฟอร์มนี้ |

## ภาพรวมการทำงานของระบบ

การตรวจจับเป็นเพียงขั้นตอนกลางของกระบวนการ คุณค่าที่แท้จริงอยู่ที่สิ่งที่เกิดขึ้นทั้งก่อนและหลังขั้นตอนดังกล่าว
ได้แก่ การนำกล้องไปยังพื้นที่เป้าหมายที่ถูกต้อง และการทำให้เจ้าหน้าที่เข้าระงับเหตุได้จริงภายหลังตรวจพบไฟ

```
   ┌──────────────┐
   │  1. DRONE    │
   │   DEPLOYED   │
   └──────┬───────┘
          │  video / stills
          ▼
   ┌──────────────┐
   │ 2. REMOTE    │
   │  INSPECTION  │
   └──────┬───────┘
          │  frames
          ▼
   ┌──────────────┐
   │ 3. COMPUTER  │  ← ส่วนที่โครงการนี้พัฒนาขึ้น
   │    VISION    │
   └──────┬───────┘
          │  detections + track IDs
          ▼
   ┌──────────────┐
   │ 4. FIRE      │
   │  DETECTED    │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 5. CONTROL   │
   │   CENTRE     │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 6. RESPONSE  │
   │  TEAM SENT   │
   └──────────────┘
```

| ลำดับ | ขั้นตอน | รายละเอียด |
|---|---|---|
| 1 | Drone deployed | ปล่อยอากาศยานบินเหนือพื้นที่สำรวจ ซึ่งครอบคลุมพื้นที่ได้รวดเร็วและเข้าถึงภูมิประเทศที่ลาดตระเวนได้ยาก |
| 2 | Remote inspection | เจ้าหน้าที่ซึ่งผ่านการฝึก ทำหน้าที่บังคับและเฝ้าติดตามอากาศยานจากสถานีภาคพื้นดิน |
| 3 | Computer vision | โมเดล YOLO26 ประมวลผลทุกเฟรมเพื่อค้นหาเปลวไฟและกลุ่มควัน |
| 4 | Fire detected | ผลการตรวจจับที่ได้รับการยืนยันจะส่งการแจ้งเตือนทันที |
| 5 | Control centre | ผู้ปฏิบัติงานประเมินการแจ้งเตือนแล้วตัดสินใจว่าจะรับมืออย่างไร |
| 6 | Response team sent | ส่งชุดปฏิบัติการ (response team) เข้าไปยังพิกัดที่ตรวจพบ |

ขั้นที่ 1, 2, 5 และ 6 เป็นส่วนของการปฏิบัติการ ซึ่งเกี่ยวข้องกับอากาศยาน บุคลากร และขั้นตอนปฏิบัติ
**ขั้นที่ 3 คือส่วนที่คลังโค้ดนี้พัฒนาขึ้น** ส่วนขั้นที่ 4 เป็นผลสืบเนื่องโดยตรงจากขั้นที่ 3
โมเดลจะค้นหาร่องรอยเชิงภาพของไฟ ได้แก่ เปลวไฟที่มองเห็นได้ กลุ่มควัน และการเปลี่ยนแปลงของสีในบริเวณที่กำลังลุกไหม้

กระบวนการพัฒนาที่นำไปสู่ขั้นตอนดังกล่าวประกอบด้วยสี่ขั้น

1. **เตรียมชุดข้อมูล** รวบรวมภาพถ่ายทางอากาศดิบ
2. **ติดป้ายกำกับและสร้างเวอร์ชันของชุดข้อมูล** กำหนดกรอบล้อมวัตถุ แล้วตรึงผลลัพธ์ไว้
3. **ฝึกโมเดล** ส่วนที่ 1
4. **ทดสอบโมเดล** ส่วนที่ 2 และส่วนที่ 3

## ขั้นที่ 1 เตรียมชุดข้อมูล

ต้นทางคือ **FLAME dataset** (*Aerial Imagery Pile burn detection using drones (UAVs)*, IEEE Dataport)
ซึ่งเป็นภาพถ่ายทางอากาศของการเผากองเชื้อเพลิงแบบมีการควบคุม ถือว่าใกล้เคียงเป้าหมายจริงมากที่สุดเท่าที่ชุดข้อมูลสาธารณะจะให้ได้

ข้อมูลที่ได้มาอยู่ในรูปแบบวิดีโอ (`.mp4`) มิใช่ภาพนิ่ง ซึ่งเป็นรูปแบบตามธรรมชาติของการบันทึกด้วยอากาศยานไร้คนขับ
แต่ไม่เหมาะสมต่อการฝึกโมเดลตรวจจับวัตถุ (object detection) ที่ต้องการภาพรายเฟรมพร้อมป้ายกำกับ

Roboflow รองรับการแปลงรูปแบบดังกล่าว โดยผู้ใช้เพียงอัปโหลดวิดีโอและกำหนดอัตราการตัดเฟรม
ระบบจะแยกวิดีโอออกเป็นภาพนิ่งรายเฟรมให้โดยอัตโนมัติ

อัตราการตัดเฟรมมีความสำคัญมากกว่าที่ปรากฏ หากกำหนดค่าสูงเกินไปจะได้ภาพที่มีลักษณะใกล้เคียงกันจำนวนมาก
ซึ่งทำให้ชุดข้อมูลมีขนาดใหญ่ขึ้นโดยไม่ได้สารสนเทศเพิ่มเติม และที่สำคัญกว่านั้น ภาพที่ใกล้เคียงกัน
อาจกระจายคร่อมเส้นแบ่งระหว่างชุดฝึก (train) กับชุดตรวจสอบ (validation)
ส่งผลให้คะแนนของชุดตรวจสอบสูงเกินจริงโดยไม่มีสัญญาณบ่งชี้ ในทางกลับกัน หากกำหนดค่าต่ำเกินไป
จะสูญเสียความหลากหลายเชิงภาพ ซึ่งเป็นปัจจัยที่ทำให้โมเดลวางนัยทั่วไป (generalise) ได้

## ขั้นที่ 2 ติดป้ายกำกับและสร้างเวอร์ชันของชุดข้อมูล

ทุกเฟรมถูกติดป้ายด้วยกรอบล้อมวัตถุ (bounding box) สำหรับงานตรวจจับวัตถุ ผ่านเครื่องมือติดป้ายกำกับของ Roboflow
โดยโครงการนี้ใช้ **คลาส (class) เดียว คือ `fire`** กำหนดกรอบครอบทุกบริเวณที่ปรากฏไฟในเฟรม

การใช้โมเดลคลาสเดียวเป็นการลดความซับซ้อนโดยเจตนา เนื่องจากหลีกเลี่ยงคำถามที่ยากที่สุดของการติดป้ายกำกับ
กล่าวคือ เปลวไฟสิ้นสุดที่ตำแหน่งใดและควันเริ่มต้นที่ตำแหน่งใด โดยแลกกับข้อจำกัดที่โมเดล
ไม่สามารถจำแนกสองสิ่งนี้ออกจากกันได้ในขั้นตอน inference อย่างไรก็ตาม สำหรับระบบเตือนภัยล่วงหน้า
ซึ่งต้องการเพียงผลลัพธ์ว่ามีการลุกไหม้เกิดขึ้น ณ ตำแหน่งใด การแลกเปลี่ยนดังกล่าวถือว่าเหมาะสม

เมื่อติดป้ายกำกับเสร็จแล้ว ชุดข้อมูลจะถูกตรึงไว้เป็น **เวอร์ชัน (version)** ซึ่งเป็นสำเนาที่แก้ไขไม่ได้
มีการแบ่ง train/validation/test และการตั้งค่าประมวลผลข้อมูลเบื้องต้น (preprocessing) เป็นของตัวเอง
เวอร์ชันดังกล่าวคือกลไกที่ทำให้การฝึกโมเดลสามารถทำซ้ำได้ (reproducible) กล่าวคือ
การเรียก `version(1)` ในภายหลังจะได้ข้อมูลชุดเดิมกลับมาเสมอ
โครงการนี้ใช้เวอร์ชันที่ 1 ของโปรเจกต์ `drone-fire-detection-byija` ในการฝึกโมเดล

## ขั้นที่ 3 การฝึกโมเดล

เมื่อได้เวอร์ชันของชุดข้อมูลที่ติดป้ายกำกับเรียบร้อยแล้ว การฝึกโมเดลประกอบด้วยสองขั้นตอน
ได้แก่ การดาวน์โหลดชุดข้อมูลจาก Roboflow และการปรับละเอียด (fine-tune) ต่อจาก checkpoint
ของ YOLO ที่ผ่านการฝึกล่วงหน้ามาแล้ว

ขั้นตอนการดาวน์โหลดชุดข้อมูล

```python
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")
dataset = project.version(1).download("yolov8")
```

จากนั้นจึงฝึกโมเดล

```python
from ultralytics import YOLO

model = YOLO("yolo26m.pt")
train_results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=800,
    plots=True,
)
```

โค้ดข้างต้นมีสองประเด็นที่ควรอธิบายเพิ่มเติม เนื่องจากแตกต่างจากบทความต้นฉบับทั้งสองประเด็น

- **`download("yolov8")` เป็นชื่อรูปแบบการจัดวางไฟล์ของชุดข้อมูล มิใช่ชื่อโมเดล**
  หมายถึงรูปแบบที่ประกอบด้วยภาพ ไฟล์ label `.txt` ตามมาตรฐาน YOLO และ `data.yaml`
  ซึ่ง YOLO26 นำไปใช้ได้โดยตรงโดยไม่ต้องปรับแก้ ทั้งนี้ ตัวเลขเวอร์ชันในสตริงดังกล่าว
  ไม่มีความเกี่ยวข้องกับเวอร์ชันของโมเดลที่นำมาฝึก
- **บทความต้นฉบับใช้ `yolov8m.pt` ผ่าน CLI `yolo task=detect mode=train`** ขณะที่คลังโค้ดนี้ฝึกด้วย **YOLO26**
  ซึ่งเป็นสถาปัตยกรรมแบบครบวงจร (end-to-end) ที่ไม่ผ่านขั้นตอน NMS โดยเรียกใช้ผ่าน Python API
  ผลในทางปฏิบัติประการหนึ่งคือ ไม่มีค่าขีดแบ่ง (threshold) `iou` ของ NMS ให้ต้องปรับอีกต่อไป

ค่า `imgsz=800` นับว่าค่อนข้างสูง และกำหนดไว้เช่นนั้นโดยเจตนา เนื่องจากไฟในภาพถ่ายทางอากาศ
มักปรากฏเป็นบริเวณขนาดเล็กเมื่อเทียบกับพื้นที่ทั้งเฟรม การลดขนาดภาพลงเหลือ 640
จึงเท่ากับตัดพิกเซลในส่วนที่สำคัญที่สุดออกไป ข้อแลกเปลี่ยน (trade-off) คือการใช้หน่วยความจำที่เพิ่มขึ้น
หากหน่วยความจำของ GPU ไม่เพียงพอ ให้ลดค่าเป็น 640 หรือเปลี่ยนไปใช้ `yolo26n.pt`

### เกณฑ์ของผลลัพธ์ที่ดี

การฝึกด้วย YOLOv8 ในบทความต้นฉบับรายงานผลบนคลาส `fire` ไว้ดังนี้

| ตัวชี้วัด | ค่าที่ได้ |
|---|---|
| Recall | 0.95 |
| mAP@50 | 0.99 |
| mAP@50-95 | 0.63 |

ควรพิจารณาค่าทั้งสามประกอบกัน มิใช่แยกพิจารณาทีละค่า ค่า Recall 0.95 และ mAP@50 0.99
บ่งชี้ว่าโมเดลตรวจพบไฟได้อย่างสม่ำเสมอ ซึ่งเป็นคุณสมบัติที่สำคัญที่สุดของระบบเตือนภัยล่วงหน้า
เนื่องจากการไม่ตรวจพบไฟที่เกิดขึ้นจริง (false negative) มีต้นทุนสูงกว่าการแจ้งเตือนที่ผิดพลาด (false positive) มาก

ส่วนค่าที่ลดลงเหลือ 0.63 ที่เกณฑ์ mAP@50-95 บ่งชี้ว่าตำแหน่งของกรอบมีความถูกต้อง *โดยประมาณ*
แต่ยังไม่แม่นยำในระดับสูง ซึ่งเป็นผลที่คาดหมายได้และไม่ถือเป็นข้อบกพร่องร้ายแรงในบริบทนี้
เนื่องจากไฟไม่มีขอบเขตที่ชัดเจนตั้งแต่ต้น คะแนนที่วัดด้วยค่า IoU สูงจึงสะท้อนความกำกวม
ของป้ายกำกับ (label ambiguity) ไม่น้อยไปกว่าข้อจำกัดของตัวโมเดลเอง

ค่าชุดนี้ควรถือเป็นเกณฑ์อ้างอิงสำหรับการเปรียบเทียบ มิใช่ค่าที่รับประกันได้
เนื่องจากผลลัพธ์ย่อมขึ้นอยู่กับเวอร์ชันของชุดข้อมูลและวิธีการแบ่งข้อมูลที่ใช้

---

### ▶ ดำเนินการ `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 1 จาก 3 (การฝึกโมเดล)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)

**การฝึกโมเดลตรวจจับ**

ส่วนนี้ดาวน์โหลดชุดข้อมูลจาก Roboflow ปรับละเอียด YOLO26 เป็นเวลา 50 epochs ตรวจสอบความแม่นยำของผลลัพธ์
ทำนายผลบนชุดทดสอบที่กันไว้ แล้วส่งออกไฟล์ weights

ขั้นตอนสุดท้ายคือการดาวน์โหลด **`best.pt`** ซึ่งเป็น checkpoint ที่ผ่านการฝึกเรียบร้อยแล้ว
ส่วนที่ 2 และส่วนที่ 3 จำเป็นต้องใช้ไฟล์ดังกล่าว และไฟล์นี้ *ไม่ได้* จัดเก็บไว้ในคลังโค้ด
จึงควรเก็บรักษาไว้ให้ดี

> **ก่อนเริ่มดำเนินการ** ให้เปลี่ยน Colab ไปใช้ runtime แบบ GPU
> (`Runtime` → `Change runtime type` → **T4 GPU**) และนำ
> [Roboflow API key](https://app.roboflow.com/settings/api) ซึ่งขอได้โดยไม่มีค่าใช้จ่าย
> ไปบันทึกไว้ในแผง 🔑 **Secrets** ของ Colab ภายใต้ชื่อ `ROBOFLOW_API_KEY`

---

## ขั้นที่ 4 การนำไปใช้งานและการทดสอบ

checkpoint ที่ผ่านการฝึกแล้วสามารถนำไปใช้งานได้หลายแนวทาง ทั้งการส่ง weights กลับไปยัง Roboflow
เพื่อให้บริการผ่าน hosted inference API ของแพลตฟอร์ม หรือการประมวลผลบนฮาร์ดแวร์ของผู้ใช้เองด้วย
[`roboflow/inference`](https://github.com/roboflow/inference) ซึ่งเป็นทางเลือกที่เหมาะสมกว่า
สำหรับสถานีภาคพื้นดินของอากาศยานไร้คนขับ เนื่องจากการรับส่งข้อมูลทุกเฟรมกับ cloud API
มีความเร็วและความเสถียรไม่เพียงพอ

อย่างไรก็ตาม หากวัตถุประสงค์คือการตรวจสอบว่าโมเดลใช้งานได้จริงหรือไม่ วิธีที่ตรงไปตรงมาที่สุด
คือการโหลด `best.pt` เข้าสู่โน้ตบุ๊กแล้วตรวจสอบผลลัพธ์ด้วยสายตา ซึ่งเป็นสิ่งที่ส่วนที่ 2
และส่วนที่ 3 ดำเนินการ โดยใช้ **[Supervision](https://supervision.roboflow.com/)**
แปลงผลลัพธ์ดิบจากโมเดลให้อยู่ในรูปของภาพและวิดีโอที่แสดงกรอบผลการตรวจจับ

## การประมวลผลภาพนิ่ง (inference)

กระบวนการสำหรับภาพนิ่งประกอบด้วยขั้นตอนไม่มาก ได้แก่ การอ่านภาพ การประมวลผลด้วยโมเดล
การแปลงผลลัพธ์เป็นออบเจกต์ `Detections` ของ Supervision และการแสดงผล

```python
import cv2
import supervision as sv
from ultralytics import YOLO

model = YOLO("best.pt")
image = cv2.imread("fire_image.png")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
```

`sv.Detections.from_ultralytics` เป็นจุดเชื่อมต่อที่มีประโยชน์ที่สุดในโค้ดชุดนี้ เนื่องจากแปลงผลลัพธ์ของ YOLO
ให้อยู่ในโครงสร้างข้อมูลที่ annotator รองรับ พร้อมทั้งนำชื่อคลาสมาด้วย
และเมื่อเปิดใช้การติดตามวัตถุ ก็จะนำหมายเลข track มาด้วยเช่นกัน

ขั้นตอนถัดไปคือการแสดงผล ซึ่งเป็นจุดที่โค้ดในบทความต้นฉบับ *ไม่สามารถทำงานได้* มิใช่เพียงล้าสมัย

```python
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

sv.plot_image(image=annotated, size=(10, 10))
```

บทความต้นฉบับส่งอาร์กิวเมนต์ `labels=` ให้ `BoxAnnotator` แต่ **อาร์กิวเมนต์ดังกล่าวถูกถอดออกตั้งแต่ supervision 0.22**
ปัจจุบันการแสดงผลแยกเป็น `BoxAnnotator` (แสดงกรอบ) และ `LabelAnnotator` (แสดงข้อความกำกับ)
ส่วนชื่อคลาสอ่านจาก `detections["class_name"]` ซึ่งได้ค่ามาจากตัวโมเดลโดยตรง
แทนการเทียบกับรายชื่อที่กำหนดไว้เอง ซึ่งมีความเสี่ยงที่จะไม่สอดคล้องกับ weights ที่ใช้งานอยู่

---

### ▶ ดำเนินการ `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 2 จาก 3 (ภาพนิ่ง)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2)

**ตรวจจับไฟบนภาพนิ่งหนึ่งภาพ**

ส่วนนี้โหลด `best.pt` ประมวลผลกับภาพตัวอย่าง `fire_image.png` ในคลังโค้ด
แสดงผลการตรวจจับด้วย Supervision แล้วบันทึกผลลัพธ์

นับเป็นวิธีที่รวดเร็วที่สุดในการตรวจสอบว่า checkpoint ที่เพิ่งฝึกเสร็จใช้งานได้จริงหรือไม่

> **จำเป็นต้องมี `best.pt`** จากส่วนที่ 1 หากดำเนินการต่อเนื่องในเซสชันเดียวกัน ระบบจะเรียกใช้ให้โดยอัตโนมัติ
> แต่หากเปิดโน้ตบุ๊กขึ้นมาใหม่แล้วเริ่มที่ส่วนนี้ ระบบจะแจ้งให้อัปโหลด
> ทั้งนี้ ส่วนนี้ไม่จำเป็นต้องใช้ GPU เนื่องจากการประมวลผลภาพเดียวใช้เวลาไม่นานบน CPU

---

## การประมวลผลวิดีโอ (inference)

การประมวลผลวิดีโอมีความซับซ้อนเพิ่มขึ้น เนื่องจากการตรวจจับเพียงอย่างเดียวไม่เพียงพอ
หากประมวลผลด้วยโมเดลตรวจจับทีละเฟรม ผลลัพธ์ที่ได้คือกรอบชุดหนึ่งต่อหนึ่งเฟรม
โดยไม่มีข้อมูลบ่งชี้ว่าไฟในเฟรมที่ 200 เป็นไฟกองเดียวกับในเฟรมที่ 199 หรือไม่
เมื่อขาดความคงอยู่ของวัตถุ (object permanence) จึงไม่สามารถนับจำนวนจุดไฟที่แยกจากกันได้
ไม่สามารถวัดระยะเวลาการลุกไหม้ของไฟแต่ละจุด และไม่สามารถติดตามการลุกลามได้

**ByteTrack** ทำหน้าที่เติมความต่อเนื่องดังกล่าว ควรทำความเข้าใจแนวคิดหลักของอัลกอริทึมนี้ไว้
เนื่องจากเป็นตัวกำหนดโครงสร้างของโค้ดที่ตามมา กล่าวคือ ตัวติดตามวัตถุ (object tracker) โดยทั่วไปจะคัดผลการตรวจจับ
ที่มีค่าความเชื่อมั่น (confidence score) ต่ำออกก่อนเข้าสู่ขั้นตอนการจับคู่ แต่ ByteTrack เก็บผลดังกล่าวไว้
แล้วนำไปจับคู่กับ track ที่สร้างไว้จากเฟรมก่อนหน้า ไฟที่ถูกควันบดบังชั่วขณะหรือถูกบังขณะอากาศยานเอียงตัว
จะมีค่าความเชื่อมั่นลดลง แต่มิได้หมายความว่าวัตถุนั้นหายไปจากฉาก
ทั้งนี้ กรอบที่มีค่าความเชื่อมั่นต่ำแต่มีตำแหน่งสอดคล้องกับ track ที่มีค่าความเชื่อมั่นสูงจากเฟรมก่อนหน้า
ย่อมมีความเป็นไปได้สูงว่าเป็นวัตถุจริง

### จุดที่โค้ดต้นฉบับใช้งานไม่ได้

เวอร์ชันที่เผยแพร่ในบทความติดตั้ง ByteTrack แบบนี้

```python
!git clone https://github.com/ifzhang/ByteTrack.git
!sed -i 's/onnx==1.8.1/onnx==1.9.0/g' requirements.txt
!pip3 install -q -r requirements.txt
!python3 setup.py -q develop
!pip install -q cython_bbox onemetric loguru lap thop
```

กล่าวคือ การคอมไพล์ YOLOX จากซอร์สโค้ด ร่วมกับ `onemetric` และ `cython_bbox`
และคำสั่ง `sed` อีกสองบรรทัดเพื่อหลีกเลี่ยงข้อบกพร่องซึ่งโครงการต้นทาง (upstream) ได้แก้ไขไปนานแล้ว
ชุดเครื่องมือดังกล่าวไม่สามารถคอมไพล์บน Python รุ่นปัจจุบันได้อีกต่อไป

อนึ่ง ขั้นตอนเหล่านี้ไม่มีความจำเป็นอีกต่อไป เนื่องจาก **ByteTrack รวมอยู่ใน Ultralytics แล้ว**
โค้ดทั้งบล็อกข้างต้นจึงลดรูปเหลือเพียงการเรียกใช้ฟังก์ชันเดียว

```python
result = model.track(
    frame,
    conf=0.1,
    persist=True,
    tracker="bytetrack.yaml",
    verbose=False,
)[0]
detections = sv.Detections.from_ultralytics(result)
detections = detections[detections.confidence >= 0.25]
```

`persist=True` ทำหน้าที่รักษาสถานะของตัวติดตามไว้ระหว่างการเรียกแต่ละครั้ง หมายเลข id จึงคงที่ข้ามเฟรม
ส่วน `from_ultralytics` อ่านหมายเลขดังกล่าวมาเก็บไว้ใน `detections.tracker_id`

ควรสังเกตค่าความเชื่อมั่นสองค่าที่ไม่เท่ากันในโค้ดข้างต้น ซึ่งเป็นการนำแนวคิดของ ByteTrack มาใช้โดยตรง
กล่าวคือ **ติดตามที่ค่า `0.1` แต่แสดงผลที่ค่า `0.25`** การป้อนเฉพาะกรอบที่มีค่าความเชื่อมั่นสูง
ให้ตัวติดตาม เท่ากับตัดผลการตรวจจับที่มีค่าความเชื่อมั่นต่ำ ซึ่งเป็นข้อมูลที่อัลกอริทึมใช้ในการกู้คืน track
ดังนั้นจึงควรติดตามจากผลการตรวจจับทั้งหมดก่อน แล้วจึงกรองในขั้นตอนการแสดงผล

อีกประเด็นหนึ่งที่ควรทราบสำหรับการเปรียบเทียบกับบทความต้นฉบับ คือลูปการเรนเดอร์ (render loop) ในบทความ
สร้างออบเจกต์ `BYTETracker` ขึ้นมาแต่ไม่เคยเรียกใช้ภายในลูป
วิดีโอผลลัพธ์ที่ได้จึงเป็นเพียงผลการตรวจจับรายเฟรมที่ไม่มีการติดตามวัตถุแต่อย่างใด

---

### ▶ ดำเนินการ `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 3 จาก 3 (วิดีโอ)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3)

**ตรวจจับและติดตามไฟตลอดทั้งวิดีโอ**

ส่วนนี้ประมวลผล `fire.mp4` ด้วย YOLO26 ร่วมกับ ByteTrack ทีละเฟรม แสดงกรอบ ป้ายกำกับ หมายเลข track
และเส้นแสดงร่องรอยการเคลื่อนที่ (trace) แล้วบันทึกเป็นวิดีโอผลลัพธ์

นอกจากนี้ยังแปลงไฟล์ผลลัพธ์เป็น H.264 ด้วย ffmpeg เพื่อให้แสดงผลได้ในโน้ตบุ๊ก
เนื่องจาก Supervision บันทึกด้วยตัวเข้ารหัส (codec) `mp4v` ซึ่งเบราว์เซอร์ไม่สามารถถอดรหัสได้
ประเด็นนี้คือสาเหตุที่วิดีโอผลลัพธ์ในบทความต้นฉบับไม่สามารถแสดงผลได้

> **จำเป็นต้องมี `best.pt`** จากส่วนที่ 1 และต้องใช้ runtime แบบ **T4 GPU**
> เนื่องจากการประมวลผลวิดีโอบน CPU ใช้เวลานานเกินกว่าจะใช้งานได้จริง

---

## ภาคผนวก การตรวจสอบสภาพแวดล้อม

เนื้อหาข้างต้นไม่มีส่วนใดที่ต้องประมวลผล เนื่องจากเอกสารฉบับนี้ทำหน้าที่เป็นแผนที่ มิใช่ไปป์ไลน์
อย่างไรก็ตาม หากต้องการตรวจสอบความพร้อมของ runtime ก่อนเริ่มดำเนินการส่วนที่ 1
สามารถประมวลผลเซลล์ด้านล่างได้

In [ ]:
import shutil
import subprocess
import sys

print(f"Python {sys.version.split()[0]}\n")

gpu = shutil.which("nvidia-smi")
if gpu:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"GPU: {out or 'ตรวจพบแล้ว'}")
else:
    print("GPU: ไม่พบ โปรดตั้งค่าที่ Runtime > Change runtime type > T4 GPU")

print("ffmpeg:", "พร้อมใช้งาน" if shutil.which("ffmpeg") else "ไม่พบ (ส่วนที่ 3 จำเป็นต้องใช้)")

for pkg in ("ultralytics", "supervision", "roboflow"):
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'ติดตั้งแล้ว')}")
    except ImportError:
        print(f"{pkg}: ยังไม่ได้ติดตั้ง (โน้ตบุ๊กจะติดตั้งให้ในส่วนที่ 1)")

## สิ่งที่ต่างจากบทความต้นฉบับ

บทความต้นฉบับเผยแพร่เมื่อเดือนกันยายน 2023 โค้ดในบทความอ้างอิงไลบรารีที่ได้รับการพัฒนาต่อมาอย่างมาก
และบางส่วนเปลี่ยนแปลงไปจนไม่สามารถทำงานได้อีกต่อไป เนื้อหาทั้งหมดในคลังโค้ดนี้จึงได้รับการปรับปรุงใหม่ ดังนี้

| | บทความ (2023) | คลังโค้ดนี้ |
|---|---|---|
| โมเดล | `yolov8m.pt` | **`yolo26m.pt`** แบบครบวงจร ไม่ผ่าน NMS |
| Ultralytics | `==8.0.20` | `>=8.4.122` |
| Supervision | `==0.1.0` | `>=0.30.0` |
| Roboflow | ไม่ได้ตรึงเวอร์ชัน | `>=1.4.1` |
| การฝึกโมเดล | CLI `yolo task=detect mode=train` | Python API ร่วมกับ `results.save_dir` |
| การวาดผลลัพธ์ | `BoxAnnotator(labels=...)` | `BoxAnnotator` + `LabelAnnotator` |
| การติดตามวัตถุ | โคลน ByteTrack แล้วคอมไพล์ YOLOX จากซอร์สโค้ด | รวมอยู่ใน Ultralytics แล้ว |
| API key | `api_key="YOUR_API_KEY"` ในเซลล์ | Colab Secrets |
| วิดีโอผลลัพธ์ | `mp4v` (เบราว์เซอร์ถอดรหัสไม่ได้) | re-encode เป็น H.264 |

นอกจากนี้ โน้ตบุ๊กชุดเดิมยังไม่สามารถแสดงผลได้ ซึ่งเกิดจากสองสาเหตุที่แยกจากกัน ได้แก่
โน้ตบุ๊กสำหรับวิดีโอมีบล็อก `metadata.widgets` ที่ขาดคีย์ `state` ซึ่งเป็นเงื่อนไขที่ทำให้ GitHub
แสดงข้อความ *"Invalid Notebook"* ส่วนโน้ตบุ๊กสำหรับการฝึกโมเดลมีขนาด 1.68 MB
ซึ่งเกือบทั้งหมดเป็นภาพผลลัพธ์ที่ฝังมาในรูปแบบ base64 และเกินขีดจำกัดการแสดงผลของ GitHub ที่ประมาณ 1 MB

ปัจจุบันโน้ตบุ๊กทุกไฟล์อยู่ในรูปแบบ `nbformat 4.5` มี cell ID ครบถ้วน และล้าง output ที่ฝังอยู่ออกทั้งหมดแล้ว

## บทสรุป

ระบบนี้ประกอบด้วยโมเดลตรวจจับ YOLO26 ที่ฝึกขึ้นเฉพาะสำหรับการค้นหาไฟในภาพถ่ายทางอากาศ
ทำงานร่วมกับตัวติดตามวัตถุที่ติดตามไฟแต่ละจุดตลอดเฟรมของวิดีโอ โดยมีวงจรการปฏิบัติงาน (operational loop) รองรับ
ซึ่งเป็นปัจจัยที่ทำให้ระบบมีประโยชน์ในทางปฏิบัติ ทั้งอากาศยานไร้คนขับที่สำรวจพื้นที่
ซึ่งไม่สามารถจัดกำลังลาดตระเวนได้ทั่วถึง และศูนย์ควบคุมที่แปลงผลการตรวจจับให้เป็นการส่งชุดปฏิบัติการเข้าพื้นที่

องค์ประกอบแต่ละส่วนไม่ได้มีความซับซ้อนเป็นพิเศษ ปัจจัยที่ทำให้แนวทางนี้ได้ผลคือ
งานที่มีต้นทุนสูงที่สุด อันได้แก่การเฝ้าสังเกตอย่างต่อเนื่องบนภูมิประเทศขนาดใหญ่และห่างไกล
เป็นงานที่โมเดลดำเนินการได้ดี ในขณะที่มนุษย์ดำเนินการได้ไม่ดีนัก

### ขั้นตอนถัดไป

เปิด **[`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)** แล้วดำเนินการตามลำดับจากบนลงล่าง

1. **[ส่วนที่ 1](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-1)** ฝึกโมเดลแล้วส่งออก `best.pt`
2. **[ส่วนที่ 2](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2)** ทดสอบกับภาพนิ่ง
3. **[ส่วนที่ 3](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3)** ทดสอบกับวิดีโอพร้อมการติดตามวัตถุ

ประเด็นที่ควรศึกษาต่อ ได้แก่ การนำโมเดลไปติดตั้งใช้งานบนสถานีภาคพื้นดินด้วย
[`roboflow/inference`](https://github.com/roboflow/inference) แทนการเรียกผ่าน cloud API
การแยกคลาส `fire` ออกเป็นคลาสเปลวไฟและคลาสควัน (ซึ่งควันสังเกตเห็นได้จากระยะไกลกว่ามาก)
และการนำหมายเลข track ไปใช้ประเมินอัตราการลุกลาม แทนการรายงานเพียงว่าตรวจพบไฟหรือไม่

### แหล่งอ้างอิงและเครดิต

- บทความต้นฉบับ: [Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/) โดย Timothy M., Roboflow Blog
- คลังโค้ดต้นทาง: [tim3in/Fire-Detection-Drone](https://github.com/tim3in/Fire-Detection-Drone)
- ชุดข้อมูล: [`drone-fire-detection-byija`](https://universe.roboflow.com/tim-4ijf0/drone-fire-detection-byija) บน Roboflow Universe

**รายการอ้างอิงชุดข้อมูล**

> Alireza Shamsoshoara, Fatemeh Afghah, Abolfazl Razi, Liming Zheng, Peter Fulé, Erik
> Blasch, November 19, 2020, "The FLAME dataset: Aerial Imagery Pile burn detection
> using drones (UAVs)", IEEE Dataport, doi: https://dx.doi.org/10.21227/qad6-r683

**รายการอ้างอิงบทความ**

> Timothy M. (Sep 19, 2023). Aerial Fire Detection with Drone Imagery and Computer
> Vision. Roboflow Blog: https://blog.roboflow.com/aerial-fire-detection/